# Demo 03 : Routing

Run the cells in order and watch. There is nothing to write.

## a · conditional-edges

We route a work order from its current priority.

In [1]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph

class Ticket(TypedDict):
    priority: str
    destination: str

def route(state: Ticket) -> Literal["page", "queue"]:
    return "page" if state["priority"] == "urgent" else "queue"

builder = StateGraph(Ticket)
builder.add_node("page", lambda s: {"destination": "engineer"})
builder.add_node("queue", lambda s: {"destination": "work queue"})
builder.add_node("route", lambda s: {})
builder.add_edge(START, "route")
builder.add_conditional_edges("route", route)
builder.add_edge("page", END)
builder.add_edge("queue", END)
graph = builder.compile()
for priority in ("urgent", "normal"):
    print(priority, "->", graph.invoke({"priority": priority, "destination": ""})["destination"])

urgent -> engineer
normal -> work queue


## b · path-map

We map router labels to node names, keeping routing decisions separate from graph labels.

In [2]:
mapped = StateGraph(Ticket)
mapped.add_node("route", lambda s: {})
mapped.add_node("page", lambda s: {"destination": "engineer"})
mapped.add_node("queue", lambda s: {"destination": "work queue"})
mapped.add_edge(START, "route")
mapped.add_conditional_edges("route", lambda s: s["priority"], {"urgent": "page", "routine": "queue"})
mapped.add_edge("page", END)
mapped.add_edge("queue", END)
mapped_graph = mapped.compile()
print("mapped:", mapped_graph.invoke({"priority": "routine", "destination": ""}))
try:
    mapped_graph.invoke({"priority": "normal", "destination": ""})
except KeyError as exc:
    print("unmapped label:", type(exc).__name__, exc)

mapped: {'priority': 'routine', 'destination': 'work queue'}
unmapped label: KeyError 'normal'


## c · recursion-limit

We cap a retrying graph at two supersteps and display LangGraph's recursion
error. Then we run the same loop with no `recursion_limit` and count the
steps before the default limit stops it.

In [3]:
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph

class Retry(TypedDict):
    attempts: int

retry = StateGraph(Retry)
retry.add_node("again", lambda s: {"attempts": s["attempts"] + 1})
retry.add_edge(START, "again")
retry.add_edge("again", "again")
try:
    retry.compile().invoke({"attempts": 0}, {"recursion_limit": 2})
except GraphRecursionError as exc:
    print(type(exc).__name__)

GraphRecursionError


In [4]:
steps = 0
try:
    for _ in retry.compile().stream({"attempts": 0}):
        steps += 1
except GraphRecursionError as exc:
    print("steps without a limit:", f"{steps:,}")
    print(str(exc).split(".")[0])

steps without a limit: 10,007
Recursion limit of 10007 reached without hitting a stop condition


## Things to try

- Add a second route label to `path_map`
- Give the retry graph a conditional exit